In [1]:

# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

import pandas as pd
from time import sleep
import datetime
from bs4 import BeautifulSoup
import os
import requests
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


In [2]:


#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'HK SFCHK' ## change to current controller name



print(f"Running {regulatorName} Web Scraping Tool v.1.12")

now=datetime.datetime.now()

filename = '{} data {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])


scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)



Running HK SFCHK Web Scraping Tool v.1.12


In [3]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------



Typology={'HK SFCHK 1':'Dealing in securities' , 'HK SFCHK 2': 'Dealing in futures contracts', 'HK SFCHK 3': 'Leveraged foreign exchange trading', 

         'HK SFCHK 4': 'Advising on securities', 'HK SFCHK 5': 'Advising on futures contracts', 'HK SFCHK 6': 'Advising on corporate finance', 

         'HK SFCHK 7': 'Providing automated trading services', 'HK SFCHK 8': 'Securities margin financing', 'HK SFCHK 9': 'Asset management', 

         'HK SFCHK 10': 'Providing credit rating services',
         'HK SFCHK 13': 'Providing depositary services for relevant CISs'}





sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [],

          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [],

          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [],

          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [],

          'Phone - Mother company': [], 'Check': []}


processdate = now.strftime('%Y-%m-%d')

In [4]:

# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict


In [5]:

# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()


In [ ]:
# %%

#------------------------------------------------ Begin_Main ----------------------------------------

url = "https://apps.sfc.hk/publicregWeb/searchByRaJson"
letters = list("ABCDEFGHIJKLMNOPQRESTUVWXYZ0123456789")  # as given
ratype_list = [1,2,3,4,5,6,7,8,9,10,13]
all_rows = []

for ratype in ratype_list:
    
    for letter in letters:
        payload = {
            "licstatus": "active",
            "roleType": "corporation",
            "ratype": str(ratype),
            "nameStartLetter": letter,
            "page": "1",
            "start": "0",
            "limit": "20",
        }
        print(f'SFO Licence: {ratype}, Letter {letter}')

        # first call to get totalCount
        resp = requests.post(url, data=payload, timeout=30,verify=False)
        resp.raise_for_status()
        data = resp.json()
        

        total = data.get("totalCount", 20)
        payload["limit"] = str(total)

        # refetch with full limit if needed
        if total > 20:
            resp = requests.post(url, data=payload, timeout=30,verify=False)
            resp.raise_for_status()
            data = resp.json()
        #print(data)

        #all_rows.extend(data.get("items", []))
        for row in data.get("items", []):
            row["_ratype"] = ratype
            all_rows.append(row)
        


for row in all_rows:

    sqldict['InternalID_1'].append(row['ceref'])
    sqldict['InternalID_1_type'].append('CE Reference')
    sqldict['Name'].append(row['name'])
    sqldict['ListProcessDate'].append(processdate)
    sqldict['RegulationType'].append('Regulated')
    sqldict['RegCtry'].append('HK')
    sqldict['RegCode'].append('SFCHK')
    sqldict['ListCode'].append(str(row["_ratype"]))
    sqldict['ListName'].append(Typology[f"{regulatorName} {row['_ratype']}"])

    if row['address']:
        #print(row['address']['fullAddress'])
        sqldict['Address_1'].append(row['address']['fullAddress'])
    else:
        sqldict['Address_1'].append('')
                                                                    


    # ceref = row["ceref"]
    # url = f"https://apps.sfc.hk/publicregWeb/corp/{ceref}/addresses"

    
    # driver.get(url)


    # wait = WebDriverWait(driver, 30)
    # try:
    #     wait.until(EC.visibility_of_element_located((By.ID, "gridemail-body")))
    #     wait.until(EC.visibility_of_element_located((By.ID, "gridwebsite-body")))
    
    #     email_div = driver.find_element(By.ID, "gridemail-body")
    #     website_div = driver.find_element(By.ID, "gridwebsite-body")

    #     email_ = email_div.text if email_div else ""
    #     website_ = website_div.text if website_div else ""

    #     #print(email_, website_)
    # except:
    #     email_,website_ = '',''
    # sqldict['Email'].append(email_)
    # sqldict['Website'].append(website_)
    sqldict['Cntry'].append('HK')
    sqldict = bourange_same_length_array(sqldict)



SFO Licence: 1, Letter A
SFO Licence: 1, Letter B
SFO Licence: 1, Letter C
SFO Licence: 1, Letter D
SFO Licence: 1, Letter E
SFO Licence: 1, Letter F
SFO Licence: 1, Letter G
SFO Licence: 1, Letter H
SFO Licence: 1, Letter I
SFO Licence: 1, Letter J
SFO Licence: 1, Letter K
SFO Licence: 1, Letter L
SFO Licence: 1, Letter M
SFO Licence: 1, Letter N
SFO Licence: 1, Letter O
SFO Licence: 1, Letter P
SFO Licence: 1, Letter Q
SFO Licence: 1, Letter R
SFO Licence: 1, Letter E
SFO Licence: 1, Letter S
SFO Licence: 1, Letter T
SFO Licence: 1, Letter U
SFO Licence: 1, Letter V
SFO Licence: 1, Letter W
SFO Licence: 1, Letter X
SFO Licence: 1, Letter Y
SFO Licence: 1, Letter Z
SFO Licence: 1, Letter 0
SFO Licence: 1, Letter 1
SFO Licence: 1, Letter 2
SFO Licence: 1, Letter 3
SFO Licence: 1, Letter 4
SFO Licence: 1, Letter 5
SFO Licence: 1, Letter 6
SFO Licence: 1, Letter 7
SFO Licence: 1, Letter 8
SFO Licence: 1, Letter 9
SFO Licence: 2, Letter A
SFO Licence: 2, Letter B
SFO Licence: 2, Letter C


In [7]:
len(all_rows)

7372

In [8]:

# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df.to_excel(filename, index=False)

# writer.save()

# writer.close()

driver.quit()

sleep(3)
    